# 04 · Centrality & Narrative Analysis
**MLS NLP Pipeline — Stage 4**

Converts graphs into ranked club metrics and detects narrative shifts.

### Metrics Computed

| Metric | What It Measures |
|--------|-----------------|
| **PageRank** | Influence — being mentioned alongside other high-profile clubs |
| **Degree centrality** | Raw number of distinct club co-mentions |
| **Eigenvector centrality** | Quality-weighted connections (being connected to important clubs) |
| **Betweenness centrality** | How often a club bridges others in the network |
| **Closeness centrality** | Average distance to all other clubs |

### Narrative Momentum
Rolling PageRank delta vs. the prior N time windows.
Labels: **rising** / **stable** / **falling** (threshold ±0.002).

### Misalignment Analysis
Gap between narrative rank (PageRank) and performance rank (points-based league standings).
Positive gap → club performs better than media attention suggests (underrated).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from pipeline.utils import get_config, PROJECT_ROOT
from pipeline.network_builder import load_graph
from pipeline.analyzer import CentralityCalculator, MomentumTracker

settings = get_config("settings")
data_dir = PROJECT_ROOT / settings["pipeline"]["data_dir"]
net_dir  = data_dir / "press" / "networks"

## 2. Computing Centrality — Single Window

In [ ]:
# Load the 2023 yearly graph
G = load_graph(net_dir / "2023" / "2023_club_cooccurrence.json")
calc = CentralityCalculator()
records = calc.compute(G, "2023", "yearly")

df_cent = pd.DataFrame([r.__dict__ for r in records]).sort_values("pagerank", ascending=False)
df_cent[["entity", "pagerank", "degree", "eigenvector", "betweenness", "closeness"]].head(15)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ("pagerank",    "PageRank",    "#e74c3c"),
    ("degree",      "Degree",      "#3498db"),
    ("betweenness", "Betweenness", "#2ecc71"),
]
for ax, (col, title, color) in zip(axes, metrics):
    top = df_cent.nlargest(10, col)
    ax.barh(top["entity"], top[col], color=color, edgecolor="white")
    ax.invert_yaxis()
    ax.set_title(f"Top 10 by {title} (2023)")
    ax.set_xlabel(col)

plt.suptitle("2023 Club Centrality Metrics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. PageRank Across All Yearly Windows

In [ ]:
all_records = []
for year in range(2018, 2025):
    G_y = load_graph(net_dir / str(year) / f"{year}_club_cooccurrence.json")
    if G_y.number_of_nodes() == 0:
        continue
    recs = calc.compute(G_y, str(year), "yearly")
    for r in recs:
        all_records.append({"entity": r.entity, "year": int(year),
                            "pagerank": r.pagerank, "degree": r.degree})

df_yr = pd.DataFrame(all_records)
df_yr["rank"] = df_yr.groupby("year")["pagerank"].rank(ascending=False).astype(int)

# Pivot to matrix
pivot = df_yr.pivot(index="entity", columns="year", values="rank").fillna(0).astype(int)
print("Narrative rank by year (1 = most prominent):")
pivot

In [ ]:
CLUB_COLORS = {
    "Inter Miami CF": "#F7B5CD", "Toronto FC": "#B81137",
    "LAFC": "#C39E6D",           "Seattle Sounders FC": "#5D9741",
    "CF Montreal": "#003DA5",    "LA Galaxy": "#00245D",
    "Philadelphia Union": "#071B2C", "Atlanta United FC": "#80000A",
}

fig, ax = plt.subplots(figsize=(13, 6))
top_clubs = df_yr.groupby("entity")["pagerank"].mean().nlargest(8).index

for club in top_clubs:
    sub = df_yr[df_yr["entity"] == club].sort_values("year")
    color = CLUB_COLORS.get(club, "#888888")
    ax.plot(sub["year"], sub["pagerank"], marker="o", lw=2, color=color, label=club)
    ax.annotate(club.split()[-1], (sub["year"].iloc[-1], sub["pagerank"].iloc[-1]),
                fontsize=7.5, color=color, xytext=(3, 0), textcoords="offset points")

ax.set_xlabel("Year")
ax.set_ylabel("PageRank")
ax.set_title("Narrative Prominence Over Time — Top 8 Clubs (Yearly PageRank)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 4. Narrative Momentum

In [ ]:
tracker = MomentumTracker(window=settings["analysis"]["momentum_window"])

# Load quarterly centrality
cent_df = pd.read_csv(data_dir / "analysis" / "press" / "centrality_club_cooccurrence.csv")
momentum_df = tracker.compute(cent_df)

# Show momentum for key clubs
key_clubs = ["Inter Miami CF", "Toronto FC", "LAFC", "Philadelphia Union"]
print("Narrative momentum (recent 8 quarters):")
(momentum_df[momentum_df["entity"].isin(key_clubs)]
 .sort_values(["entity", "time_window"])
 .tail(32)[["entity", "time_window", "pagerank", "momentum_delta", "momentum_label"]])

## 5. Quarter-over-Quarter Spikes

In [ ]:
spikes = pd.read_csv(data_dir / "analysis" / "press" / "narrative_spikes.csv")
drops  = pd.read_csv(data_dir / "analysis" / "press" / "narrative_drops.csv")

print("Top 10 narrative spikes (QoQ PageRank increase):")
print(spikes.head(10)[["entity", "time_window", "delta", "pct_change"]].to_string(index=False))

print("\nTop 10 narrative drops:")
print(drops.head(10)[["entity", "time_window", "delta", "pct_change"]].to_string(index=False))

## 6. Misalignment Analysis — Narrative vs. Performance

In [ ]:
perf = pd.read_csv(data_dir / "performance" / "mls_standings.csv")
narr = df_yr.copy()

perf["perf_rank"] = perf.groupby("year")["points"].rank(ascending=False).astype(int)
merged = narr.merge(perf.rename(columns={"club": "entity"})[["entity", "year", "points", "perf_rank"]],
                    on=["entity", "year"], how="inner")
merged["gap"] = merged["perf_rank"] - merged["rank"]  # positive = underrated

avg_gap = (merged.groupby("entity")["gap"]
           .mean()
           .sort_values(ascending=False)
           .reset_index()
           .rename(columns={"gap": "avg_gap"}))

fig, ax = plt.subplots(figsize=(13, 8))
colors = ["#27ae60" if v > 0 else "#e74c3c" for v in avg_gap["avg_gap"]]
ax.barh(avg_gap["entity"], avg_gap["avg_gap"], color=colors)
ax.axvline(0, color="black", lw=1.2)
ax.set_xlabel("Avg Gap (performance rank − narrative rank)
Positive = underrated  |  Negative = overhyped")
ax.set_title("Narrative vs. Performance Misalignment (2018–2024 avg)")
plt.tight_layout()
plt.show()

print("\nTop 5 most underrated clubs (performance >> narrative):")
print(avg_gap[avg_gap["avg_gap"] > 0].head(5).to_string(index=False))
print("\nTop 5 most overhyped clubs (narrative >> performance):")
print(avg_gap[avg_gap["avg_gap"] < 0].tail(5).to_string(index=False))